In [ ]:
# One-cell: Denoise with ffmpeg arnndn -> transcribe with whisper
# Uses your folder: /content/drive/MyDrive/ffmpeg/wav

import os
import sys
import glob
import subprocess
from pathlib import Path

# ---- Config ----
DRIVE_WAV_DIR = "/content/drive/MyDrive/ffmpeg/wav"   # <- your directory
CLEAN_OUT = "/content/cleaned"                        # cleaned wavs go here
TRANS_OUT = "/content/transcripts"                    # transcripts go here
MODEL_PATH = "/content/cleaning_model.rnnn"           # change if model is elsewhere
WHISPER_MODEL = "medium"                              # "tiny","base","small","medium","large"

# ---- Ensure Drive is mounted (Colab) ----
if not os.path.exists("/content/drive"):
    try:
        from google.colab import drive
        print("Mounting Google Drive...")
        drive.mount("/content/drive")
    except Exception as e:
        print("Could not auto-mount drive. If running locally, ensure /content/drive is accessible.")
        print("Error:", e)

# ---- Basic checks & create output dirs ----
if not os.path.isdir(DRIVE_WAV_DIR):
    raise FileNotFoundError(f"Input WAV folder not found: {DRIVE_WAV_DIR}")

os.makedirs(CLEAN_OUT, exist_ok=True)
os.makedirs(TRANS_OUT, exist_ok=True)

# ---- Verify RNNoise model file exists ----
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"RNNoise model not found at {MODEL_PATH}. Upload it to /content or update MODEL_PATH.")

# ---- Ensure ffmpeg available ----
def run_cmd(cmd):
    """Run a command list, return (rc, stdout, stderr)."""
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return proc.returncode, proc.stdout, proc.stderr

rc, out, err = run_cmd(["ffmpeg", "-version"])
if rc != 0:
    raise RuntimeError("ffmpeg not found or not working. Make sure ffmpeg is installed in the runtime.\nffmpeg error:\n" + err)

# ---- Ensure whisper is installed and import it ----
try:
    import whisper
except Exception:
    print("Installing openai-whisper (may take a minute)...")
    rc, out, err = run_cmd([sys.executable, "-m", "pip", "install", "openai-whisper", "torch", "--quiet"])
    if rc != 0:
        print("pip install failed:\n", err)
        raise RuntimeError("Failed to install whisper/torch.")
    import whisper

# ---- Load whisper model once ----
print(f"Loading Whisper model '{WHISPER_MODEL}' (this can take a moment)...")
wh = whisper.load_model(WHISPER_MODEL)

# ---- Gather WAVs ----
wav_files = sorted(glob.glob(os.path.join(DRIVE_WAV_DIR, "*.wav")))
print(f"Found {len(wav_files)} WAV files in {DRIVE_WAV_DIR}")

if len(wav_files) == 0:
    raise RuntimeError("No .wav files found. Check the input folder path and that files are .wav")

# ---- Processing loop ----
for wav in wav_files:
    name = Path(wav).stem
    cleaned = os.path.join(CLEAN_OUT, f"{name}_clean.wav")
    transcript_file = os.path.join(TRANS_OUT, f"{name}.txt")

    print(f"\n--- Processing '{name}' ---")
    print("Cleaning with ffmpeg arnndn filter...")

    # ffmpeg arnndn expects m='path' (quoted). Use list form to avoid shell quoting issues.
    filter_arg = f"arnndn=m='{MODEL_PATH}'"
    cmd = [
        "ffmpeg", "-y",
        "-i", wav,
        "-af", filter_arg,
        cleaned
    ]

    rc, out, err = run_cmd(cmd)
    if rc != 0:
        print("ffmpeg failed for:", wav)
        print("ffmpeg stderr (truncated):\n", err.splitlines()[-20:])
        # continue to next file instead of crashing entire loop
        continue
    else:
        print("Saved cleaned file ->", cleaned)

    # Transcribe with whisper
    try:
        print("Transcribing cleaned audio with Whisper...")
        result = wh.transcribe(cleaned)
        text = result.get("text", "").strip()
        with open(transcript_file, "w", encoding="utf-8") as f:
            f.write(text)
        print("Transcript saved ->", transcript_file)
    except Exception as e:
        print("Whisper transcription failed for:", cleaned)
        print("Error:", e)
        continue

print("\nALL DONE. Cleaned files in:", CLEAN_OUT)
print("Transcripts in:", TRANS_OUT)


Mounting Google Drive...
Mounted at /content/drive
Installing openai-whisper (may take a minute)...
Loading Whisper model 'medium' (this can take a moment)...


100%|█████████████████████████████████████| 1.42G/1.42G [00:18<00:00, 81.4MiB/s]


Found 28 WAV files in /content/drive/MyDrive/ffmpeg/wav

--- Processing '3_part1' ---
Cleaning with ffmpeg arnndn filter...
Saved cleaned file -> /content/cleaned/3_part1_clean.wav
Transcribing cleaned audio with Whisper...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcript saved -> /content/transcripts/3_part1.txt

--- Processing '3_part1_clean' ---
Cleaning with ffmpeg arnndn filter...
ffmpeg failed for: /content/drive/MyDrive/ffmpeg/wav/3_part1_clean.wav
ffmpeg stderr (truncated):
 ['ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers', '  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)', '  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-libra

...

Transcript saved -> /content/transcripts/7_part2.txt

--- Processing '7_part2_clean' ---
Cleaning with ffmpeg arnndn filter...
ffmpeg failed for: /content/drive/MyDrive/ffmpeg/wav/7_part2_clean.wav
ffmpeg stderr (truncated):
 ['ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers', '  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)', '  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-libra

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcript saved -> /content/transcripts/7_part3.txt

--- Processing '7_part3_clean' ---
Cleaning with ffmpeg arnndn filter...
ffmpeg failed for: /content/drive/MyDrive/ffmpeg/wav/7_part3_clean.wav
ffmpeg stderr (truncated):
 ['ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers', '  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)', '  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-libra

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcript saved -> /content/transcripts/7_part4.txt

ALL DONE. Cleaned files in: /content/cleaned
Transcripts in: /content/transcripts
